# NeMo Curator + NeMo AutoModel: Pharma PDF → non-instruction fine-tuning

**Colab lab | CPU for data preparation, optional T4 for LoRA training**

This notebook is a NeMo rewrite of the uploaded *Non-instruction Causal LLM Fine-tuning / Domain-Adaptive Continued Pretraining* notebook. It works with the supplied **Metformin-Lipid-Therapy-Knowledge.pdf** (upload that PDF when prompted).

### What is genuinely NeMo here?

- **PyMuPDF** extracts text from a PDF. NeMo Curator's text API does not itself replace a dedicated PDF parser.
- **NeMo Curator** uses its real `Pipeline`, `JsonlReader`, `Modify`, `ScoreFilter`, and `JsonlWriter` components to clean and filter text on the **CPU**.
- **NeMo AutoModel** uses `NeMoAutoModelForCausalLM.from_pretrained`, its **LoRA `PeftConfig`**, and the `automodel` recipe to train on a **T4 GPU**. The recipe uses a small custom *raw-text next-token* dataset factory, **not** instruction-response SFT.
- **Hugging Face/PEFT** are still used for loading a saved adapter for inference. NeMo AutoModel deliberately supports this checkpoint format.

**Important:** The six-page training PDF alone is much too small to establish a meaningful medical-domain model, so section 2b optionally adds more public study text (as PDFs). This is a learning experiment; do not rely on any medical conclusions. The T4 section was run end to end on a Colab T4 (Python 3.13, torch 2.11, nemo-automodel 0.6.0, transformers 5.12) on 2026-10-21. Library versions move quickly, so the install cells pin the versions that were tested.

**NVIDIA references:** [Curator cleaning](https://docs.nvidia.com/nemo/curator/v26.09/curate-text/process-data/content-processing/text-cleaning), [Curator filtering](https://docs.nvidia.com/nemo/curator/v26.09/curate-text/process-data/quality-assessment), [AutoModel PEFT recipes](https://docs.nvidia.com/nemo/automodel/latest/recipes-e2e-examples/sft-peft), [AutoModel custom datasets](https://docs.nvidia.com/nemo/automodel/latest/datasets/text-dataset).

### What you will learn

1. How a real **NeMo Curator** pipeline cleans and filters text on the CPU.
2. How raw text becomes next-token training examples: `input_ids` plus labels **shifted by one** (section 7).
3. How one **NeMo AutoModel YAML recipe** describes an entire LoRA run (section 8).
4. How to read the live training log and judge a run with **perplexity on training vs held-out text** (sections 10-11).

### Files this notebook creates (under `/content/nemo_pharma_lab`)

- `raw/pdf_pages.jsonl`: one JSON record per PDF page
- `curated/`: NeMo Curator output
- `dataset/train.jsonl`, `dataset/validation.jsonl`: page-level split
- `pharma_causal_dataset.py`: the dataset factory the YAML points to
- `pharma_t4_lora.yaml`: the training recipe
- `tinyllama_safetensors/`: the downloaded base model
- `checkpoints/`: the trained LoRA adapter
- `train.log`: the full, unfiltered training log

## 1. Select runtime

For **cleaning only**: Colab → **Runtime → Change runtime type → CPU**.

For **cleaning + training in one session**: choose **T4 GPU** now. Curator will still execute its text processing on the CPU; the GPU is used in the AutoModel training step.

The PDF is uploaded directly into your Colab runtime; nothing in this notebook requires Google Drive permissions.

In [1]:
# Install only the CPU text backend of NeMo Curator (no RAPIDS/CUDA stack).
# %pip (not !pip) is the correct notebook cell form.
%pip -q install "nemo-curator[text_cpu]==1.4.0" "pymupdf>=1.24" "pyyaml>=6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 7.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 12.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 10.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 47.9 MB/s 

In [3]:
import os, re, json, sys, unicodedata
from pathlib import Path
import pymupdf
import torch

ROOT = Path('/content/nemo_pharma_lab') if Path('/content').exists() else Path.cwd()/'nemo_pharma_lab'
RAW_DIR = ROOT/'raw'; CURATED_DIR = ROOT/'curated'; DATA_DIR = ROOT/'dataset'
for directory in [ROOT, RAW_DIR, CURATED_DIR, DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Working directory:', ROOT)

Python: 3.13.15
GPU available: True
GPU: Tesla T4
Working directory: /content/nemo_pharma_lab


## 2. Upload the pharmaceutical PDF

Run the cell and select **Metformin-Lipid-Therapy-Knowledge.pdf**. This is a file picker. If a PDF is already in `/content` (for example after a runtime restart) the picker is skipped. You may also select **several PDFs at once**: every uploaded PDF goes through the same pipeline.

In [4]:
from google.colab import files

# Skip the file picker when a PDF is already in /content (re-runs, runtime restarts)
if not list(Path('/content').glob('*.pdf')):
    files.upload()          # select Metformin-Lipid-Therapy-Knowledge.pdf (and any other PDFs)
UPLOADED_PDFS = sorted(Path('/content').glob('*.pdf'))
if not UPLOADED_PDFS:
    raise ValueError('Upload at least one PDF (Metformin-Lipid-Therapy-Knowledge.pdf).')
print('Using uploaded PDFs:', [p.name for p in UPLOADED_PDFS])

Saving Metformin-Lipid-Therapy-Knowledge.pdf to Metformin-Lipid-Therapy-Knowledge.pdf
Using uploaded PDFs: ['Metformin-Lipid-Therapy-Knowledge.pdf']


## 2b. Add more study material (recommended)

One six-page PDF is far too small for fine-tuning to show a real effect (section 11 shows why). This cell fetches the introductions of several related medical topics from English Wikipedia (text licensed [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)) and renders each one as a **real PDF** in `study_pdfs/`, so the pipeline treats them exactly like your uploaded PDF.

- Good habit: keep the **source and licence** of every training document.
- One article (**Atorvastatin**) is held out as the validation set: the model never trains on it, so it measures whether learning generalizes to an unseen document.
- Set `ADD_STUDY_PDFS = False` to train on the uploaded PDF alone. If there is no internet access, the cell skips itself.
- Wikipedia is not a medical authority; this is practice data for learning the NeMo workflow, not clinical advice.

In [5]:
ADD_STUDY_PDFS = True
STUDY_TOPICS = ['Metformin', 'Type 2 diabetes', 'Statin', 'Dyslipidemia', 'AMP-activated protein kinase',
                'Lactic acidosis', 'Biguanide', 'Atorvastatin']
HELD_OUT_TITLE = 'Atorvastatin'          # validation document: never used for training
MAX_CHARS_PER_ARTICLE = 14000            # keeps the training run to a few minutes on a T4
STUDY_DIR = ROOT/'study_pdfs'

def fetch_article(title):
    import requests
    r = requests.get('https://en.wikipedia.org/w/api.php', timeout=30,
                     headers={'User-Agent': 'nemo-learning-notebook/1.0 (educational use)'},
                     params={'action': 'query', 'prop': 'extracts', 'explaintext': 1,
                             'exsectionformat': 'plain', 'redirects': 1, 'titles': title,
                             'format': 'json'})
    r.raise_for_status()
    page = next(iter(r.json()['query']['pages'].values()))
    return page.get('extract', '')

def text_to_pdf(text, out_path, chars_per_page=3000):
    import html
    paragraphs = [p.strip() for p in text.split('\n') if p.strip()]
    pages, buf = [], ''
    for para in paragraphs:
        while len(para) > chars_per_page:               # split very long paragraphs at a space
            cut = para.rfind(' ', 0, chars_per_page)
            cut = cut if cut > 0 else chars_per_page
            if buf:
                pages.append(buf); buf = ''
            pages.append(para[:cut]); para = para[cut:].strip()
        if len(buf) + len(para) > chars_per_page and buf:
            pages.append(buf); buf = ''
        buf += para + '\n'
    if buf.strip():
        pages.append(buf)
    doc = pymupdf.open()
    for body in pages:
        page = doc.new_page()
        markup = ''.join(f'<p>{html.escape(line)}</p>' for line in body.split('\n') if line.strip())
        page.insert_htmlbox(pymupdf.Rect(50, 50, 545, 792), markup,
                            css='body {font-family: sans-serif; font-size: 10pt;} p {margin: 0 0 6pt 0;}',
                            scale_low=0.6)
    doc.save(str(out_path)); doc.close()
    return len(pages)

if ADD_STUDY_PDFS:
    STUDY_DIR.mkdir(parents=True, exist_ok=True)
    for title in STUDY_TOPICS:
        out = STUDY_DIR/f"study_{title.replace(' ', '_')}.pdf"
        if out.exists():
            print(f'{title}: already built ({out.name})')
            continue
        try:
            text = fetch_article(title)
            text = text[:MAX_CHARS_PER_ARTICLE].rsplit(' ', 1)[0] if len(text) > MAX_CHARS_PER_ARTICLE else text
            n_pages = text_to_pdf(text, out)
            print(f'{title}: {len(text.split())} words -> {out.name} ({n_pages} pages)')
        except Exception as exc:                          # offline, blocked, or article missing
            print(f'Skipped {title}: {exc}')
    print('Study PDFs:', sorted(p.name for p in STUDY_DIR.glob('*.pdf')))
else:
    print('ADD_STUDY_PDFS is False: using only the uploaded PDF(s).')

Metformin: 2058 words -> study_Metformin.pdf (5 pages)
Type 2 diabetes: 2172 words -> study_Type_2_diabetes.pdf (6 pages)
Statin: 2077 words -> study_Statin.pdf (6 pages)
Dyslipidemia: 1709 words -> study_Dyslipidemia.pdf (6 pages)
AMP-activated protein kinase: 2079 words -> study_AMP-activated_protein_kinase.pdf (5 pages)
Lactic acidosis: 1788 words -> study_Lactic_acidosis.pdf (5 pages)
Biguanide: 594 words -> study_Biguanide.pdf (2 pages)
Atorvastatin: 2006 words -> study_Atorvastatin.pdf (6 pages)
Study PDFs: ['study_AMP-activated_protein_kinase.pdf', 'study_Atorvastatin.pdf', 'study_Biguanide.pdf', 'study_Dyslipidemia.pdf', 'study_Lactic_acidosis.pdf', 'study_Metformin.pdf', 'study_Statin.pdf', 'study_Type_2_diabetes.pdf']


## 3. Extract PDF text (PyMuPDF, not NeMo)

Every page becomes one JSONL record, so its provenance is retained. You can inspect the raw output **before any cleaning**.

In [6]:
from collections import Counter

# Uploaded PDFs plus the optional study PDFs from section 2b
PDF_FILES = list(UPLOADED_PDFS)
if 'STUDY_DIR' in globals() and STUDY_DIR.exists():
    PDF_FILES += sorted(STUDY_DIR.glob('*.pdf'))

raw_records = []
for pdf_path in PDF_FILES:
    with pymupdf.open(pdf_path) as pdf:
        for page_no, page in enumerate(pdf, start=1):
            text = page.get_text('text')  # preserve text order of the original PDF
            if text and text.strip():
                raw_records.append({'id': f'{pdf_path.stem}_p{page_no:02d}',
                                    'source': pdf_path.stem,
                                    'source_page': page_no,
                                    'text': text})
RAW_JSONL = RAW_DIR/'pdf_pages.jsonl'
with RAW_JSONL.open('w', encoding='utf-8') as fp:
    for record in raw_records:
        fp.write(json.dumps(record, ensure_ascii=False)+'\n')
print(f'Extracted {len(raw_records)} pages from {len(PDF_FILES)} PDF(s). Raw JSONL: {RAW_JSONL}')
for source, count in Counter(r['source'] for r in raw_records).items():
    print(f'  {source}: {count} pages')
print('\n--- BEFORE CURATION: first 850 characters of the first page ---\n')
print(raw_records[0]['text'][:850])

Extracted 47 pages from 9 PDF(s). Raw JSONL: /content/nemo_pharma_lab/raw/pdf_pages.jsonl
  Metformin-Lipid-Therapy-Knowledge: 6 pages
  study_AMP-activated_protein_kinase: 5 pages
  study_Atorvastatin: 6 pages
  study_Biguanide: 2 pages
  study_Dyslipidemia: 6 pages
  study_Lactic_acidosis: 5 pages
  study_Metformin: 5 pages
  study_Statin: 6 pages
  study_Type_2_diabetes: 6 pages

--- BEFORE CURATION: first 850 characters of the first page ---

Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway an

## 4. Build a real NeMo Curator pipeline

- `UnicodeReformatter` corrects encoding artifacts.
- `NewlineNormalizer` standardizes excessive blank lines.
- The **custom NeMo `DocumentModifier`** removes document-specific headings, footers, and explicitly labelled Q&A samples from **this particular teaching PDF**. Generic Curator cleaning cannot know automatically which domain-specific content is undesired.
- `WordCountFilter` rejects very short records.

This demo deliberately does **not** launch GPU RAPIDS deduplication for a few dozen pages; it would add complexity without useful acceleration.

In [7]:
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.text.io.reader import JsonlReader
from nemo_curator.stages.text.io.writer import JsonlWriter
from nemo_curator.stages.text.modifiers import Modify, DocumentModifier
from nemo_curator.stages.text.modifiers.unicode import UnicodeReformatter
from nemo_curator.stages.text.modifiers.string import NewlineNormalizer
from nemo_curator.stages.text.filters import ScoreFilter
from nemo_curator.stages.text.filters.heuristic import WordCountFilter

class PharmaPDFCleaner(DocumentModifier):
    """Domain-specific text cleaner used *inside* a NeMo Curator Modify stage."""
    def __init__(self):
        super().__init__()

    def modify_document(self, text: str) -> str:
        text = unicodedata.normalize('NFKC', text)
        text = text.replace('\u200b', '').replace('\ufeff', '')
        text = re.sub(r'(?m)^\s*Pharma Domain Training Data - Page \d+\s*$', '', text)
        text = re.sub(r'(?m)^Page \d+ - [^\n]*$', '', text)
        text = re.sub(r'(?m)^Pharma-domain corpus extension[^\n]*$', '', text)
        # This file includes instructional training examples which do not belong
        # in the non-instruction RAW-TEXT demonstration. Page 3 also has a
        # following 'Important distinction' section; preserve that section.
        sample_headers = (
            'Fine-tuning style sample|Mechanism-aware data sample|'
            'Instruction-response training sample|Fine-tuning sample|Training examples'
        )
        text = re.sub(
            rf'(?s)\n(?:{sample_headers})\s*\n.*?(?=\nImportant distinction\s*\n|\Z)',
            '', text,
        )
        text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
        text = re.sub(r'[ \t]+', ' ', text)
        # Fix PDF hard-wrapped lines, retaining no spurious page layout tokens.
        text = re.sub(r'\s*\n\s*', ' ', text)
        return re.sub(r' +', ' ', text).strip()

pipe = Pipeline(name='pharma_pdf_curation')
pipe.add_stage(JsonlReader(file_paths=str(RAW_JSONL), fields=['id','text','source','source_page']))
pipe.add_stage(Modify(UnicodeReformatter()))
pipe.add_stage(Modify(NewlineNormalizer()))
pipe.add_stage(Modify(PharmaPDFCleaner()))
pipe.add_stage(ScoreFilter(filter_obj=WordCountFilter(min_words=80),
                          text_field='text', score_field='word_count'))
pipe.add_stage(JsonlWriter(path=str(CURATED_DIR)))
print('Stages:', [type(stage).__name__ for stage in pipe.stages])
from nemo_curator.backends.xenna import XennaExecutor

executor = XennaExecutor(
    config={
        "execution_mode": "batch",
        "max_workers_per_stage": 1,
    }
)

result = pipe.run(executor=executor)
print('Curator completed; output directory:', CURATED_DIR)

2026-10-23 04:38:14.220 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'jsonl_reader' to pipeline 'pharma_pdf_curation'
2026-10-23 04:38:14.221 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'UnicodeReformatter' to pipeline 'pharma_pdf_curation'
2026-10-23 04:38:14.222 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'NewlineNormalizer' to pipeline 'pharma_pdf_curation'
2026-10-23 04:38:14.225 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'PharmaPDFCleaner' to pipeline 'pharma_pdf_curation'
2026-10-23 04:38:14.226 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'word_count' to pipeline 'pharma_pdf_curation'
2026-10-23 04:38:14.229 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'jsonl_writer' to pipeline 'pharma_pdf_curation'


Stages: ['JsonlReader', 'Modify', 'Modify', 'Modify', 'ScoreFilter', 'JsonlWriter']


2026-10-23 04:38:14.625 | INFO     | nemo_curator.pipeline.pipeline:build:123 - Planning pipeline: pharma_pdf_curation
2026-10-23 04:38:14.626 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:184 - Decomposing composite stage: jsonl_reader
2026-10-23 04:38:14.627 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:198 - Expanded 'jsonl_reader' into 2 execution stages
2026-10-23 04:38:14.660 | INFO     | nemo_curator.backends.xenna.executor:execute:162 - Execution mode: BATCH
2026-10-23 04:38:14,739	WARNING authentication_token_setup.py:85 -- Token authentication is enabled for this Ray cluster. Set RAY_AUTH_MODE=disabled to opt out. For more information, see https://docs.ray.io/en/latest/ray-security/token-auth.html
2026-10-23 04:38:14,741	INFO authentication_token_setup.py:50 -- Generated new authentication token and saved to /root/.ray/auth_token
2026-10-23 04:38:30,146	INFO worker.py:2024 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:

Curator completed; output directory: /content/nemo_pharma_lab/curated


## 5. Show original → cleaned text and quality counts

The page information is kept so you can audit which text survived. `NeMo Curator` produces JSONL shards rather than a single in-memory list.

In [8]:
all_jsonl = sorted(CURATED_DIR.rglob('*.jsonl'))
print('NeMo Curator output shards:', [str(p.relative_to(CURATED_DIR)) for p in all_jsonl])
curated = []
for path in all_jsonl:
    with path.open(encoding='utf-8') as fp:
        for line in fp:
            if line.strip():
                curated.append(json.loads(line))
curated.sort(key=lambda row: (row['source'], int(row['source_page'])))
assert curated, 'The Curator output is empty. Inspect its logs or lower WordCountFilter.min_words.'
print('Raw pages:', len(raw_records), '| Curated pages:', len(curated))

raw_count = Counter(r['source'] for r in raw_records)
kept_count = Counter(r['source'] for r in curated)
for source in raw_count:
    words = sum(len(r['text'].split()) for r in curated if r['source'] == source)
    print(f'  {source}: {kept_count[source]}/{raw_count[source]} pages kept, {words} words')

# Before/after demo on one page of the first uploaded PDF
demo_source = UPLOADED_PDFS[0].stem
demo_page = 3 if any(x['source'] == demo_source and x['source_page'] == 3 for x in curated) else 1
before = next((x for x in raw_records if x['source'] == demo_source and x['source_page'] == demo_page), None)
after = next((x for x in curated if x['source'] == demo_source and x['source_page'] == demo_page), None)
if before and after:
    print(f'\n--- BEFORE cleaning ({demo_source}, page {demo_page}, last 700 chars) ---\n')
    print(before['text'][-700:])
    print(f'\n--- AFTER NeMo Curator ({demo_source}, page {demo_page}, last 700 chars) ---\n')
    print(after['text'][-700:])
print('\nResidual QA markers:',
      [(x['source'], x['source_page'], marker) for x in curated
       for marker in ['Instruction:', 'Question:', 'Response:'] if marker in x['text']])

NeMo Curator output shards: ['f090ccb1493d.jsonl']
Raw pages: 47 | Curated pages: 46
  Metformin-Lipid-Therapy-Knowledge: 6/6 pages kept, 1509 words
  study_AMP-activated_protein_kinase: 5/5 pages kept, 2079 words
  study_Atorvastatin: 6/6 pages kept, 2006 words
  study_Biguanide: 2/2 pages kept, 594 words
  study_Dyslipidemia: 5/6 pages kept, 1652 words
  study_Lactic_acidosis: 5/5 pages kept, 1788 words
  study_Metformin: 5/5 pages kept, 2058 words
  study_Statin: 6/6 pages kept, 2078 words
  study_Type_2_diabetes: 6/6 pages kept, 2173 words

--- BEFORE cleaning (Metformin-Lipid-Therapy-Knowledge, page 3, last 700 chars) ---

icine alone? Answer: Atorvastatin
reduces cholesterol synthesis in the liver, while ezetimibe reduces cholesterol absorption in the intestine. Because the two
medicines act on different points of cholesterol homeostasis, the combination can produce a stronger LDL-C reduction
than monotherapy in many patients.
Important distinction
Atorvastatin and ezetimibe are 

## 6. Build RAW-TEXT train/validation records

**No Q&A/chat format is created.** Each JSONL row is `{"text": "...", "source": ..., "source_page": ...}`. With the study PDFs present, one **whole document** (Atorvastatin) is held out for validation: nothing about it is seen during training, and paragraphs from the same page are never mixed across splits. With a single uploaded PDF the final page is held out instead, which is statistically weak and only useful for illustrating the code.

In [9]:
assert len(curated) >= 2, 'Need at least two curated pages.'
sources = sorted({r['source'] for r in curated})
held_out_source = f"study_{HELD_OUT_TITLE.replace(' ', '_')}" if 'HELD_OUT_TITLE' in globals() else None
if held_out_source in sources and len(sources) > 1:
    # Hold out a WHOLE document: a much more honest test than holding out one page
    val_docs = [r for r in curated if r['source'] == held_out_source]
    train_docs = [r for r in curated if r['source'] != held_out_source]
    print('Held-out validation document:', held_out_source)
else:
    # Single-PDF fallback: hold out only the final page
    last = max(curated, key=lambda r: (r['source'], int(r['source_page'])))
    key = (last['source'], int(last['source_page']))
    val_docs = [r for r in curated if (r['source'], int(r['source_page'])) == key]
    train_docs = [r for r in curated if (r['source'], int(r['source_page'])) != key]
    print('Single-PDF fallback: held-out page', key)
for split, records in [('train', train_docs), ('validation', val_docs)]:
    with (DATA_DIR/f'{split}.jsonl').open('w', encoding='utf-8') as fp:
        for r in records:
            fp.write(json.dumps({'text': r['text'], 'source': r['source'],
                                 'source_page': r['source_page']}, ensure_ascii=False)+'\n')
for split, records in [('Training', train_docs), ('Validation', val_docs)]:
    words = sum(len(r['text'].split()) for r in records)
    print(f"{split}: {len(records)} pages, {words} words, sources: {sorted({r['source'] for r in records})}")
print('Example training JSON:', json.dumps({'text': train_docs[0]['text'][:250]}, ensure_ascii=False))

Held-out validation document: study_Atorvastatin
Training: 40 pages, 13931 words, sources: ['Metformin-Lipid-Therapy-Knowledge', 'study_AMP-activated_protein_kinase', 'study_Biguanide', 'study_Dyslipidemia', 'study_Lactic_acidosis', 'study_Metformin', 'study_Statin', 'study_Type_2_diabetes']
Validation: 6 pages, 2006 words, sources: ['study_Atorvastatin']
Example training JSON: {"text": "Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid ox"}


# PART B — NeMo AutoModel: optional T4 LoRA training

**CPU users:** stop here; your corpus is already prepared. Run the final download cell to export it.

**T4 users:** continue. This part uses the **NeMo AutoModel training CLI**, not `transformers.Trainer`. NeMo AutoModel may install substantial dependencies. If its install fails in the current Colab image, the recommended reproducible alternative is an NVIDIA preconfigured AutoModel container on an external GPU machine.

We retain the notebook's original non-chat TinyLlama base model (1.1B) to make comparisons meaningful. The configuration uses **FP32** weights, **rank-8 LoRA** (about 0.6% of parameters are trainable), batch size 1, and 192-token chunks as a conservative starting point for a T4. A T4 does **not** support BF16 acceleration, and pure FP16 training gave NaN loss in testing, so FP32 is the safe choice here. On BF16-capable GPUs (A100, L4, H100) you would normally switch to `bfloat16`.

In [10]:
# Make the T4 section an explicit choice. CPU runs skip training safely.
import sys, json
from pathlib import Path
import torch

# Paths are re-derived here so this part still works after a runtime restart
ROOT = Path('/content/nemo_pharma_lab') if Path('/content').exists() else Path.cwd()/'nemo_pharma_lab'
RAW_DIR = ROOT/'raw'; CURATED_DIR = ROOT/'curated'; DATA_DIR = ROOT/'dataset'
RAW_JSONL = RAW_DIR/'pdf_pages.jsonl'

RUN_TRAINING = torch.cuda.is_available()
HF_MODEL_ID = 'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T'
MODEL_NAME = HF_MODEL_ID      # becomes the local download folder in section 8
SEQ_LEN = 192
print('Run NeMo AutoModel training:', RUN_TRAINING)
if RUN_TRAINING:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU-only run: skipping all model downloads and training.')

Run NeMo AutoModel training: True
GPU: Tesla T4


In [11]:
# NeMo AutoModel is installed only when GPU training is requested.
# Avoid installing nemo-automodel[cuda] or Curator's text_cuda12 extra on Colab T4.
# The version is pinned to the one this notebook was tested with (Colab T4, Oct 2026).
if RUN_TRAINING:
    import subprocess, importlib.metadata as md
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nemo-automodel==0.6.0'], check=True)
    for pkg in ['nemo-automodel', 'nemo-curator', 'torch', 'transformers', 'peft']:
        try:
            print(f'{pkg}: {md.version(pkg)}')
        except md.PackageNotFoundError:
            print(f'{pkg}: not installed')

nemo-automodel: 0.6.0
nemo-curator: 1.4.0
torch: 2.11.0+cu130
transformers: 5.12.1
peft: 0.21.2


## 7. Create a custom *causal language modeling* dataset factory

The original notebook tokenizes and packs text before passing it to Hugging Face `Trainer`. NeMo AutoModel lets you put this logic in a reusable dataset factory and reference it from YAML.

For non-instruction training the **whole text is the target**: at every position the model must predict the **next** token. So each block holds two lists of equal length:

- `input_ids` = tokens 0 .. n-1
- `labels` = tokens 1 .. n (the same text **shifted left by one**)

> **Why the shift matters.** NeMo's masked cross-entropy compares logits and labels position by position; it does **not** shift for you. (Hugging Face models shift internally when you pass `labels=input_ids`, which is why that convention is common.) An earlier version of this notebook used `labels == input_ids`: the model was asked to "predict" the token it was already reading. Loss started at 14.6 (worse than random guessing, about 10.4) and the tuned model fell into repetition loops. With the shift, the starting loss is about 1.4.

Padded labels are masked with `-100` by the collator. This is **not** `SFTSingleTurnPreprocessor` with answer-only loss.

In [12]:
DATASET_PY = ROOT/'pharma_causal_dataset.py'
DATASET_PY.write_text(r'''from pathlib import Path
import json
from datasets import Dataset


def make_pharma_dataset(tokenizer, split, data_dir, seq_length=192, min_block_tokens=24):
    """NeMo AutoModel dataset factory for raw-text next-token training.

    Each block holds input_ids (tokens 0..n-1) and labels (tokens 1..n): the label
    at position i is the token that FOLLOWS input position i. NeMo's masked
    cross-entropy compares logits and labels position by position, so the shift
    must already be done here.
    """
    path = Path(data_dir) / f"{split}.jsonl"
    rows = []
    with path.open(encoding="utf-8") as fp:
        for line in fp:
            if not line.strip():
                continue
            text = json.loads(line)["text"].strip()
            if not text:
                continue
            ending = tokenizer.eos_token or ""
            token_ids = tokenizer.encode(text + ending, add_special_tokens=False)
            for pos in range(0, len(token_ids) - 1, seq_length):
                chunk = token_ids[pos:pos + seq_length + 1]  # +1 token serves as the last label
                inputs, targets = chunk[:-1], chunk[1:]
                if len(inputs) >= min_block_tokens:
                    rows.append({
                        "input_ids": inputs,
                        "attention_mask": [1] * len(inputs),
                        "labels": targets,  # next-token targets, already shifted
                    })
    if not rows:
        raise ValueError(f"No token blocks in {path}; reduce min_block_tokens")
    print(f"[{split}] {len(rows)} causal token blocks from {path}")
    return Dataset.from_list(rows)
''', encoding='utf-8')
print('Dataset builder:', DATASET_PY)
print(DATASET_PY.read_text()[:1900])

Dataset builder: /content/nemo_pharma_lab/pharma_causal_dataset.py
from pathlib import Path
import json
from datasets import Dataset


def make_pharma_dataset(tokenizer, split, data_dir, seq_length=192, min_block_tokens=24):
    """NeMo AutoModel dataset factory for raw-text next-token training.

    Each block holds input_ids (tokens 0..n-1) and labels (tokens 1..n): the label
    at position i is the token that FOLLOWS input position i. NeMo's masked
    cross-entropy compares logits and labels position by position, so the shift
    must already be done here.
    """
    path = Path(data_dir) / f"{split}.jsonl"
    rows = []
    with path.open(encoding="utf-8") as fp:
        for line in fp:
            if not line.strip():
                continue
            text = json.loads(line)["text"].strip()
            if not text:
                continue
            ending = tokenizer.eos_token or ""
            token_ids = tokenizer.encode(text + ending, add_special_tokens=False)
        

## 8. Create NeMo AutoModel's YAML training recipe

This is the main code difference: instead of manually constructing `BitsAndBytesConfig → LoraConfig → Trainer → train()`, we declare model, LoRA, dataset, loss, optimizer, and GPU strategy in one YAML. For a single T4, tensor parallelism is **1**, not 2.

The default split is page-based; train and validation are **not** identical.

### How to read the recipe

| Key | What it controls |
|---|---|
| `recipe` | which training loop to run (here: next-token fine-tuning) |
| `step_scheduler` | batch sizes, epochs, and how often to validate and checkpoint. Global batch 4 with local batch 1 means 4 gradient-accumulation steps |
| `model` | which pretrained model to load and in what precision (`torch_dtype`) |
| `peft` | LoRA settings: `dim` = adapter rank (size), `alpha` = scaling, `target_modules` = which layers get adapters (`*_proj` = all attention and MLP projections), `dropout` |
| `distributed` | how the model is split across GPUs (one GPU here, so nothing is split) plus the `mp_policy` precision policy |
| `loss_fn` | masked cross-entropy: positions labelled `-100` are ignored |
| `dataset`, `validation_dataset` | `_target_` is `file.py:function`, the factory from section 7, followed by its arguments |
| `dataloader` | how examples are batched |
| `optimizer`, `clip_grad_norm` | AdamW with learning rate `lr` (around 1e-4 to 2e-4 is common for LoRA) and gradient clipping |
| `checkpoint` | where the LoRA adapter is saved |

In [13]:
import yaml
CONFIG = ROOT/'pharma_t4_lora.yaml'
MODEL_DIR = ROOT/'tinyllama_safetensors'

if RUN_TRAINING:
    # NeMo AutoModel loads safetensors, so download only what is needed
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=HF_MODEL_ID, local_dir=str(MODEL_DIR),
                      allow_patterns=['model.safetensors', 'config.json', 'generation_config.json',
                                      'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json',
                                      'special_tokens_map.json'])
    assert (MODEL_DIR/'model.safetensors').exists()
    # TinyLlama ships without a pad token; set one explicitly so NeMo does not default to id 0
    from transformers import AutoTokenizer
    _tok = AutoTokenizer.from_pretrained(str(MODEL_DIR))
    if _tok.pad_token is None:
        _tok.pad_token = _tok.unk_token or _tok.eos_token
        _tok.save_pretrained(str(MODEL_DIR))
    MODEL_NAME = str(MODEL_DIR)   # later cells load the model from this local folder

recipe = {
  'recipe': 'TrainFinetuneRecipeForNextTokenPrediction',
  'step_scheduler': {'global_batch_size': 4, 'local_batch_size': 1,
                     'ckpt_every_steps': 50, 'val_every_steps': 25, 'num_epochs': 2},
  'dist_env': {'backend': 'nccl', 'timeout_minutes': 5},
  'model': {'_target_': 'nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained',
            'pretrained_model_name_or_path': MODEL_NAME,
            'torch_dtype': 'float32'},
  'peft': {'_target_': 'nemo_automodel.components._peft.lora.PeftConfig',
           'target_modules': '*_proj', 'dim': 8, 'alpha': 16,
           'dropout': 0.05, 'use_triton': False},
  'distributed': {'strategy': 'fsdp2', 'tp_size': 1, 'cp_size': 1,
                  # fp32 compute: pure fp16 gave NaN loss here, and a T4 has no native bf16.
                  # Without an explicit policy NeMo would fall back to bf16 and crash on a T4.
                  'mp_policy': {'_target_': 'torch.distributed.fsdp.MixedPrecisionPolicy',
                                'param_dtype': 'float32', 'reduce_dtype': 'float32',
                                'output_dtype': None, 'cast_forward_inputs': False}},
  'loss_fn': {'_target_': 'nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy'},
  'dataset': {'_target_': f'{DATASET_PY}:make_pharma_dataset',
              'split': 'train', 'data_dir': str(DATA_DIR), 'seq_length': SEQ_LEN},
  'validation_dataset': {'_target_': f'{DATASET_PY}:make_pharma_dataset',
              'split': 'validation', 'data_dir': str(DATA_DIR), 'seq_length': SEQ_LEN},
  'dataloader': {'_target_': 'torchdata.stateful_dataloader.StatefulDataLoader',
                 'collate_fn': 'nemo_automodel.components.datasets.utils.default_collater',
                 'shuffle': False},
  'validation_dataloader': {'_target_': 'torchdata.stateful_dataloader.StatefulDataLoader',
                 'collate_fn': 'nemo_automodel.components.datasets.utils.default_collater'},
  'optimizer': {'_target_': 'torch.optim.AdamW', 'lr': 2e-4, 'weight_decay': 0.01},
  'clip_grad_norm': {'max_norm': 1.0},
  'checkpoint': {'enabled': True, 'checkpoint_dir': str(ROOT/'checkpoints'),
                 'model_save_format': 'safetensors', 'save_consolidated': False},
}
CONFIG.write_text(yaml.safe_dump(recipe, sort_keys=False), encoding='utf-8')
print(CONFIG.read_text())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

recipe: TrainFinetuneRecipeForNextTokenPrediction
step_scheduler:
  global_batch_size: 4
  local_batch_size: 1
  ckpt_every_steps: 50
  val_every_steps: 25
  num_epochs: 2
dist_env:
  backend: nccl
  timeout_minutes: 5
model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: /content/nemo_pharma_lab/tinyllama_safetensors
  torch_dtype: float32
peft:
  _target_: nemo_automodel.components._peft.lora.PeftConfig
  target_modules: '*_proj'
  dim: 8
  alpha: 16
  dropout: 0.05
  use_triton: false
distributed:
  strategy: fsdp2
  tp_size: 1
  cp_size: 1
  mp_policy:
    _target_: torch.distributed.fsdp.MixedPrecisionPolicy
    param_dtype: float32
    reduce_dtype: float32
    output_dtype: null
    cast_forward_inputs: false
loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy
dataset:
  _target_: /content/nemo_pharma_lab/pharma_causal_dataset.py:make_pharma_dataset
  split: train
  data_dir: /content/nemo_pharma_

## 9. Check training input (T4 only)

Inspect the custom factory **before** using GPU training. This check catches empty data, wrong labels, and the temptation to convert text into instruction/response pairs.

In [14]:
if RUN_TRAINING:
    import importlib.util
    from transformers import AutoTokenizer
    spec = importlib.util.spec_from_file_location('pharma_causal_dataset', DATASET_PY)
    dsmod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(dsmod)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    train_preview = dsmod.make_pharma_dataset(tok,'train', str(DATA_DIR), seq_length=SEQ_LEN)
    val_preview = dsmod.make_pharma_dataset(tok,'validation', str(DATA_DIR), seq_length=SEQ_LEN)
    sample = train_preview[0]
    assert sample['labels'][:-1] == sample['input_ids'][1:], 'labels must be the inputs shifted left by one'
    assert set(sample) == {'input_ids','attention_mask','labels'}
    print('train blocks:',len(train_preview), 'validation blocks:',len(val_preview))
    print('Tokens in first block:',len(sample['input_ids']))
    print('Decoded first block: ',tok.decode(sample['input_ids'][:90]))
    print('Input tokens 0-5 :', tok.convert_ids_to_tokens(sample['input_ids'][:6]))
    print('Label tokens 0-5 :', tok.convert_ids_to_tokens(sample['labels'][:6]), '(each label = the NEXT input token)')
else:
    print('Skipped tokenizer/model download in CPU-only mode.')

[train] 142 causal token blocks from /content/nemo_pharma_lab/dataset/train.jsonl
[validation] 21 causal token blocks from /content/nemo_pharma_lab/dataset/validation.jsonl
train blocks: 142 validation blocks: 21
Tokens in first block: 192
Decoded first block:  Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin
Input tokens 0-5 : ['▁Met', 'form', 'in', '▁is', '▁one', '▁of']
Label tokens 0-5 : ['form', 'in', '▁is', '▁one', '▁of', '▁the'] (each label = the NEXT input token)


## 10. Train with the REAL NeMo AutoModel CLI

`automodel config.yaml` is NeMo AutoModel training, **not** Hugging Face `Trainer` hidden behind a NeMo label. The recipe controls LoRA and saves a PEFT adapter checkpoint.

The log streams live and is filtered to the important lines (the full log is saved to `train.log`). A step line looks like `step 7 | epoch 1 | loss 0.93 | grad_norm 2.3 | lr 2e-4 | ...`.

- **loss** should start around 1-3 for a healthy next-token setup and trend down. It is noisy here (a batch of 2). A starting loss of 10+ or `nan` means a bug (wrong labels, wrong precision).
- **steps** = blocks x epochs / global batch size; the log prints `Max train steps` at the start so you know the length of the run.
- **grad_norm** is the gradient size before clipping; sudden huge values signal instability.

If you hit an out-of-memory error, first reduce `SEQ_LEN` (and regenerate the YAML), or switch to a smaller compatible pretrained model.

In [15]:
if RUN_TRAINING:
    import subprocess, shutil, time
    # A leftover checkpoint makes NeMo resume at the final step and skip training entirely
    shutil.rmtree(ROOT/'checkpoints', ignore_errors=True)
    LOG = ROOT/'train.log'          # the full, unfiltered log is kept here
    interesting = ('| step ', 'val', 'Trainable parameters', 'Total parameters',
                   'causal token blocks', 'Max train steps', 'checkpoint', 'Traceback', 'Error')
    noise = ('not documented', 'Training:')   # library docstring spam and tqdm progress bars

    print('Launching: automodel', CONFIG)
    start = time.time()
    tail = []
    with subprocess.Popen(['automodel', str(CONFIG)], cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc, LOG.open('w') as log:
        for line in proc.stdout:                 # streamed live, not only at the end
            log.write(line)
            line = line.rstrip()
            tail = (tail + [line])[-25:]
            if any(k in line for k in interesting) and not any(n in line for n in noise):
                print(line.split(' | root | ')[-1][:190])
    print(f'Exit code: {proc.returncode}  ({time.time() - start:.0f}s)  full log: {LOG}')
    if proc.returncode != 0:
        print('\n'.join(tail))
        raise RuntimeError('Training failed. See the last log lines above.')
else:
    print('No GPU: data curation is complete; NeMo AutoModel training not run.')

Launching: automodel /content/nemo_pharma_lab/pharma_t4_lora.yaml
  val_every_steps: 25
validation_dataset:
  split: validation
validation_dataloader:
checkpoint:
  checkpoint_dir: /content/nemo_pharma_lab/checkpoints
Trainable parameters: 6,307,840
Total parameters: 1,106,356,224
Trainable parameters percentage: 0.57%
- Max train steps: 72
- Checkpoint retention: disabled; keeping all checkpoints (checkpoint.max_recent_checkpoints=None)
[train] 142 causal token blocks from /content/nemo_pharma_lab/dataset/train.jsonl
[validation] 21 causal token blocks from /content/nemo_pharma_lab/dataset/validation.jsonl
step 0 | epoch 0 | loss 1.6816 | grad_norm 2.0063 | lr 2.00e-04 | mem 5.06 GiB | tps 65.83(65.83/gpu) | num_label_tokens 747
step 25 | epoch 0 | loss 1.5513 | grad_norm 1.3208 | lr 2.00e-04 | mem 5.11 GiB | tps 165.08(165.08/gpu) | num_label_tokens 749
2026-10-23 04:45:51 | INFO | nemo_automodel.recipes.base_recipe | Saving checkpoint to /content/nemo_pharma_lab/checkpoints/epoch_0_

## 11. Compare base vs NeMo fine-tuned

The same LoRA adapter is switched **off** (base model) and **on** (fine-tuned) on the same weights, so only the adapter differs.

**Perplexity** is `exp(average next-token loss)`: roughly "how many tokens the model is torn between at each step". Lower is better. Two splits are measured:

- **train**: pages the model was trained on. It should drop a lot (the model is memorizing them).
- **validation**: a page the model never saw. This is the number that shows whether it learned anything that generalizes.

**What the tested runs showed.** With the section 2b study PDFs (142 training blocks), train perplexity fell from 6.80 to 3.61 and validation, an entire unseen document, improved from 6.62 to 6.40: a real but modest gain. With only the six-page PDF (14 blocks, and a different, much smaller validation set, so the numbers are not directly comparable), train perplexity fell from 7.10 to 2.19 while validation got **worse** (9.01 to 9.48). That is classic **overfitting**: the model memorizes what it trained on. More data turned memorizing into a little generalizing. Small validation sets are also noisy, so treat differences of a few percent with care.

Then one prompt is completed by both models with greedy decoding. The model learns text continuation, not medical Q&A. Only run after successful GPU training.

In [16]:
if RUN_TRAINING:
    # PEFT needs a recent torchao to import cleanly on current Colab images
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchao>=0.16.0'], check=True)

    import gc, math, re, textwrap, importlib.util, torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    PROMPT = 'Metformin is a biguanide antihyperglycemic medicine'
    REPETITION_PENALTY = 1.0   # try 1.2 if the text loops

    adapter_files = sorted((ROOT/'checkpoints').rglob('adapter_model.safetensors'),
                           key=lambda x: int(re.search(r'step_(\d+)', str(x)).group(1))
                           if re.search(r'step_(\d+)', str(x)) else -1)
    if not adapter_files:
        raise FileNotFoundError('No NeMo LoRA checkpoint found. Check training logs.')
    adapter_dir = adapter_files[-1].parent

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to('cuda')
    model = PeftModel.from_pretrained(base, adapter_dir).eval()

    # ---- 1. Numbers: perplexity = exp(mean next-token loss), lower is better ----
    spec = importlib.util.spec_from_file_location('pharma_causal_dataset', DATASET_PY)
    dsmod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(dsmod)

    def perplexity(split):
        data = dsmod.make_pharma_dataset(tokenizer, split, str(DATA_DIR), seq_length=SEQ_LEN)
        total, count = 0.0, 0
        for row in data:
            ids = torch.tensor([row['input_ids']], device='cuda')
            labels = torch.tensor(row['labels'], device='cuda')
            with torch.no_grad():
                logits = model(input_ids=ids).logits[0].float()
            total += torch.nn.functional.cross_entropy(logits, labels, reduction='sum').item()
            count += labels.numel()
        return math.exp(total / count)

    rows = []
    for split in ['train', 'validation']:
        with model.disable_adapter():           # adapter off = the untouched base model
            rows.append((split, perplexity(split)))
        rows[-1] += (perplexity(split),)        # adapter on = fine-tuned
    print('\nPerplexity (lower is better)')
    print(f"{'split':<12}{'base':>10}{'fine-tuned':>14}")
    for split, b, t in rows:
        print(f'{split:<12}{b:>10.2f}{t:>14.2f}')
    print('validation = a page the model never trained on; train = pages it did see')

    # ---- 2. One answer, same prompt, same weights, greedy decoding ----
    inputs = tokenizer(PROMPT, return_tensors='pt').to('cuda')

    def complete():
        with torch.no_grad():
            out = model.generate(**inputs, do_sample=False, max_new_tokens=65,
                                 repetition_penalty=REPETITION_PENALTY,
                                 pad_token_id=tokenizer.eos_token_id)
        return textwrap.fill(tokenizer.decode(out[0], skip_special_tokens=True), width=60)

    with model.disable_adapter():
        base_text = complete()
    tuned_text = complete()
    print('\nAdapter:', adapter_dir)
    print('Prompt:', PROMPT)
    print('\n--- BASE MODEL ---\n' + base_text)
    print('\n--- FINE-TUNED (NeMo LoRA) ---\n' + tuned_text)

    del model, base, inputs
    gc.collect()
    torch.cuda.empty_cache()
else:
    print('Comparison requires a completed T4 training run.')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[train] 142 causal token blocks from /content/nemo_pharma_lab/dataset/train.jsonl
[train] 142 causal token blocks from /content/nemo_pharma_lab/dataset/train.jsonl
[validation] 21 causal token blocks from /content/nemo_pharma_lab/dataset/validation.jsonl
[validation] 21 causal token blocks from /content/nemo_pharma_lab/dataset/validation.jsonl


[transformers] Both `max_new_tokens` (=65) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Perplexity (lower is better)
split             base    fine-tuned
train             6.80          3.61
validation        6.62          6.40
validation = a page the model never trained on; train = pages it did see


[transformers] Both `max_new_tokens` (=65) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Adapter: /content/nemo_pharma_lab/checkpoints/epoch_1_step_71/model
Prompt: Metformin is a biguanide antihyperglycemic medicine

--- BASE MODEL ---
Metformin is a biguanide antihyperglycemic medicine that is
used to treat type 2 diabetes. It is also used to treat
gestational diabetes. Metformin is a member of the biguanide
class of drugs. It is a sulfonylurea. Sulfonylureas are a
class of drugs that are used

--- FINE-TUNED (NeMo LoRA) ---
Metformin is a biguanide antihyperglycemic medicine that is
used to treat type 2 diabetes mellitus. It is also used to
treat polycystic ovary syndrome, and to lower the risk of
cardiovascular disease in people with type 2 diabetes.
Metformin is a member of the biguanide class


## 12. Export the data and recipe as a ZIP

Exports: raw PDF text JSONL, NeMo Curator output, train/validation JSONL, custom NeMo AutoModel dataset builder, the YAML recipe, and the study PDFs (if you built them in section 2b). The YAML contains absolute `/content/...` paths, so edit them if you unzip elsewhere. **Model checkpoint adapters** are separate (and remain under `checkpoints/`).

In [17]:
import shutil
from pathlib import Path

# Re-derive paths so this cell still works after a runtime restart
if 'ROOT' not in globals():
    ROOT = Path('/content/nemo_pharma_lab') if Path('/content').exists() else Path.cwd()/'nemo_pharma_lab'
RAW_DIR = ROOT/'raw'; CURATED_DIR = ROOT/'curated'; DATA_DIR = ROOT/'dataset'
DATASET_PY = ROOT/'pharma_causal_dataset.py'; CONFIG = ROOT/'pharma_t4_lora.yaml'
RAW_JSONL = RAW_DIR/'pdf_pages.jsonl'

required = [DATA_DIR/'train.jsonl', DATA_DIR/'validation.jsonl', DATASET_PY, CONFIG, RAW_JSONL]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Run the earlier cells first; missing: ' + ', '.join(missing))

EXPORT = ROOT/'exports'
shutil.rmtree(EXPORT, ignore_errors=True)  # drop stale files from earlier runs
EXPORT.mkdir(parents=True)
for name in ['train.jsonl', 'validation.jsonl']:
    shutil.copy2(DATA_DIR/name, EXPORT/name)
for file in [DATASET_PY, CONFIG, RAW_JSONL]:
    shutil.copy2(file, EXPORT/file.name)
if (ROOT/'study_pdfs').exists():
    shutil.copytree(ROOT/'study_pdfs', EXPORT/'study_pdfs')
for i, file in enumerate(sorted(CURATED_DIR.rglob('*.jsonl')), 1):
    shutil.copy2(file, EXPORT/f'curated_shard_{i}.jsonl')
ZIP_PATH = ROOT.parent/'pharma_nemo_curator_automodel_data.zip'
shutil.make_archive(str(ZIP_PATH.with_suffix('')), 'zip', EXPORT)
print('Created:', ZIP_PATH)
try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except ImportError:
    print('Not running in Colab: copy the ZIP from the path above.')

Created: /content/pharma_nemo_curator_automodel_data.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What changed from the uploaded notebook?

| Existing Hugging Face notebook | NeMo-based version |
|---|---|
| `fitz.open()` to read PDF | **Same**: PDF parsing still needed |
| `clean_pdf_text()` with extensive hand-written regex | `NeMo Curator Pipeline` + standard `UnicodeReformatter`, `NewlineNormalizer`, and one small PDF-specific `DocumentModifier` |
| Manual paragraph length / duplicate checks | Curator `WordCountFilter` and curation-stage output. No GPU dedup for six pages |
| `Dataset.from_list()`, `train_test_split()` | Page-level hold-out then JSONL; raw causal-token `make_pharma_dataset()` registered in AutoModel YAML |
| `AutoModelForCausalLM` + `BitsAndBytesConfig` | `NeMoAutoModelForCausalLM.from_pretrained` in a recipe; **no QLoRA** in this low-resource demo |
| `LoraConfig` + `get_peft_model` | NeMo AutoModel `PeftConfig` in YAML |
| `TrainingArguments` + `Trainer(...).train()` | `automodel pharma_t4_lora.yaml` |
| PEFT adapter save / reload | AutoModel saves the adapter; reload via Hugging Face `PeftModel` |

**Why not QLoRA?** On a 16 GB T4 with a 1.1B base model, normal LoRA in FP32 is simpler to teach (FP16 gave NaN loss here and the T4 has no native bf16). The original notebook uses 4-bit QLoRA; switching to standard LoRA is an intentional **memory/precision tradeoff**, not a claim that NeMo requires it. NVIDIA documents QLoRA support in AutoModel, but the 4-bit backend increases installation complexity for Colab.

**When should NeMo Curator be preferred?** For reusable audited stages, large collections, neural filtering and distributed processing. For one small six-page PDF it may use *more* code and setup than the original regex pipeline: the learning benefit is to practice a scalable API, **not** immediate speed.

## Where this stops, and what changes at scale

- **One GPU only.** The log says NeMo skips parallelization when the world size is 1, so the multi-GPU machinery (FSDP2 sharding, tensor parallelism via `tp_size`, context parallelism via `cp_size`) is configured but not exercised. See the NVIDIA AutoModel docs for how to launch the same style of recipe on several GPUs.
- **Precision.** BF16-capable GPUs would use `bfloat16` instead of FP32.
- **Data.** The six-page PDF alone is 14 blocks (roughly 2,700 tokens); with the section 2b study PDFs it is 142 blocks (about 27,000 tokens). Real domain adaptation needs millions of tokens, and that is where Curator's deduplication, filtering and classifier stages start to matter.
- **Model size.** A 1.1B model trains easily on a T4. Much larger models need QLoRA or several GPUs.

## Exercises

1. In section 8, change `lr` and `num_epochs`, rerun sections 8-11, and look for settings that **lower validation perplexity**, not just train.
2. Change the LoRA rank `dim` (4, 16, 32) and watch the trainable-parameter count in the log.
3. Narrow `target_modules` (for example only attention projections) and compare quality and speed.
4. Set `ADD_STUDY_PDFS = False` in section 2b, rerun everything, and watch validation perplexity get worse (overfitting). Then add your own topics to `STUDY_TOPICS`, or upload more PDFs in section 2, and watch the gap shrink.
5. Break it on purpose: in the dataset factory set `labels` equal to `inputs` again and watch the starting loss jump above 10.
6. Set `REPETITION_PENALTY = 1.2` in section 11 and compare the generated text.

In [ ]:
!find /content/nemo_pharma_lab/checkpoints -type f

/content/nemo_pharma_lab/checkpoints/training.jsonl
/content/nemo_pharma_lab/checkpoints/epoch_1_step_71/optim/__0_0.distcp
/content/nemo_pharma_lab/checkpoints/epoch_1_step_71/optim/.metadata
/content/nemo_pharma_lab/checkpoints/epoch_1_step_71/model/tokenizer.model
/content/nemo_pharma_lab/checkpoints/epoch_1_step_71/model/automodel_peft_config.json
/content/nemo_pharma_lab/checkpoints/epoch_1_step_71/model/tokenizer_config.json
/content/nemo_pharma_lab/checkpoints/epoch_1_step_71/model/special_tokens_map.json
/content/nemo_pharma_lab/checkpoints/epoch_1_step_71/model/adapter_config.json
/content/nemo_pharma_lab/checkpoints/epoch_1_step_71/model/adapter_model.safetensors
/content/nemo_pharma_lab/checkpoints/epoch_1_step_71/model/tokenizer.json
/content/nemo_pharma_lab/checkpoints/epoch_1_step_71/losses.json
/content/nemo_pharma_lab/checkpoints/epoch_1_step_71/rng/rng_global_rank_0.pt
/content/nemo_pharma_lab/checkpoints/epoch_1_step_71/step_scheduler.pt
/content/nemo_pharma_lab/check